# 05 · DeBERTa-v3-base: main model and E2–E7

**Runtime:** A100 GPU (bf16). **Run after:** 01 and 03. **Time:** ~4–5 h on A100 in total. The sections are independent and resumable, so you can run them across several sessions. Finished runs are skipped.

| Section | Experiment | Runs |
|---|---|---|
| A | E2 DeBERTa (50K, CE), the shared reference for E3 and E4 | 3 |
| B | E3 weighted CE, focal loss (50K) | 6 |
| C | E4 claim-only, retrieved evidence (50K) | 6 |
| D | Main model: full FEVER, 3 seeds, saved to Drive (E1, E5 zero-shot, E6) | 3 |
| E | E5 SciFact transfer: learning curve and from-scratch control | 15 |
| F | E7 paraphrase augmentation (full data + 15K paraphrases) | 3 |

In [ ]:
# Setup: Drive, code, packages, paths  (expected runtime: A100)
from google.colab import drive
drive.mount('/content/drive')
import os, sys
BASE = '/content/drive/MyDrive/neural-evidence'
CODE = '/content/drive/MyDrive/Colab Notebooks/neural-evidence'   # your uploaded code folder
REPO_URL = ''
if not os.path.exists(f'{CODE}/ne') and REPO_URL:
    os.system(f'git clone -q {REPO_URL} /content/code'); CODE = '/content/code'
assert os.path.exists(f'{CODE}/ne'), f'Code not found at {CODE}'
os.chdir(CODE); sys.path.insert(0, CODE)
!pip -q install -r requirements.txt sentencepiece protobuf
PROC, RESULTS, CACHE = f'{BASE}/data/processed', f'{BASE}/results', f'{BASE}/cache'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
from ne import runtime
from ne.data import read_jsonl, write_jsonl, eval_sets
runtime.setup()
if runtime.gpu_kind() not in 'A100'.split('/') and 'A100' != 'CPU':
    print('WARNING: expected A100; this runtime is', runtime.gpu_kind(), '- it will still work, just at a different speed/cost')
EVALS = eval_sets(PROC); print('eval sets:', list(EVALS))

In [ ]:
from ne.train import RunConfig, run, evaluate_saved
SEEDS = (42, 43, 44)
MODEL = 'microsoft/deberta-v3-base'
TEST = {k: v for k, v in EVALS.items() if k != 'fever_val'}
if 'fever_test_para' not in TEST:
    print('WARNING: robustness sets missing; run notebook 03 first (section D can be re-scored later)')
BASE_CFG = dict(model_name=MODEL, train_file=f'{PROC}/fever/train.jsonl', val_file=f'{PROC}/fever/val.jsonl',
                eval_files=TEST, epochs=3, lr=2e-5, max_length=128, results_dir=RESULTS, scratch_dir='/content/ckpt')
SUB = dict(BASE_CFG, train_subset=50_000)

## A · E2: DeBERTa on the 50K subset (cross-entropy)

In [ ]:
for s in SEEDS:
    run(RunConfig(exp='E2', name='deberta_v3_base', seed=s, **SUB))

## B · E3: loss and class imbalance (cross-entropy reference = section A)

In [ ]:
for loss in ('weighted_ce', 'focal'):
    for s in SEEDS:
        run(RunConfig(exp='E3', name=f'deberta_{loss}', loss=loss, seed=s, **SUB))

## C · E4: evidence ablation (gold evidence reference = section A)

In [ ]:
for ev in ('none', 'retrieved'):
    for s in SEEDS:
        run(RunConfig(exp='E4', name='deberta_claim_only' if ev == 'none' else 'deberta_retrieved',
                      evidence=ev, seed=s, **SUB))

## D · Main model: full FEVER training data, saved for E5 and E6

In [ ]:
for s in SEEDS:
    run(RunConfig(exp='MAIN', name='deberta_full', seed=s, save_model=True, **BASE_CFG))

In [ ]:
# If the robustness sets were created after section D ran, score the saved models on them now
for s in SEEDS:
    cfg = RunConfig(exp='MAIN', name='deberta_full', seed=s, **BASE_CFG)
    missing = {k: v for k, v in TEST.items() if not os.path.exists(f'{cfg.run_dir}/metrics_{k}.json')}
    if missing:
        evaluate_saved(f'{cfg.run_dir}/model', cfg, missing)

## E · E5: SciFact transfer

SciFact train claims (809) are used for fine-tuning and SciFact dev claims (300) for testing, so no test claim is ever trained on.
* `zeroshot_from_fever`: the main model with no SciFact training
* `fever_then_scifact_{10,25,50,100}pct`: the main model fine-tuned on a fraction of SciFact train
* `scifact_only_100pct`: DeBERTa fine-tuned on SciFact alone (shows what FEVER pre-training adds)

In [ ]:
sf = read_jsonl(f'{PROC}/scifact/test.jsonl')
write_jsonl([r for r in sf if r['split'] == 'scifact_train'], f'{PROC}/scifact/ft_train.jsonl')
write_jsonl([r for r in sf if r['split'] == 'scifact_dev'], f'{PROC}/scifact/ft_test.jsonl')
SF_EVAL = {'scifact_dev': f'{PROC}/scifact/ft_test.jsonl'}
E5 = dict(train_file=f'{PROC}/scifact/ft_train.jsonl', val_file=None, eval_files=SF_EVAL, max_length=256,
          epochs=10, lr=2e-5, select_best=False, results_dir=RESULTS, scratch_dir='/content/ckpt')
for s in SEEDS:
    main_dir = RunConfig(exp='MAIN', name='deberta_full', seed=s, **BASE_CFG).run_dir + '/model'
    zs = RunConfig(exp='E5', name='zeroshot_from_fever', model_name=main_dir, seed=s, **E5)
    if not os.path.exists(f'{zs.run_dir}/metrics_scifact_dev.json'):
        evaluate_saved(main_dir, zs, SF_EVAL)
    for frac in (10, 25, 50, 100):
        n = round(809 * frac / 100)
        run(RunConfig(exp='E5', name=f'fever_then_scifact_{frac}pct', model_name=main_dir,
                      train_subset=0 if frac == 100 else n, seed=s, **E5))
    run(RunConfig(exp='E5', name='scifact_only_100pct', model_name=MODEL, seed=s, **E5))

## F · E7: paraphrase augmentation (compare with MAIN on `fever_test_para` and `fever_test`)

In [ ]:
aug = f'{PROC}/robustness/train_para_15k.jsonl'
assert os.path.exists(aug), 'run notebook 03 first'
for s in SEEDS:
    run(RunConfig(exp='E7', name='deberta_full_para_aug', augment_file=aug, seed=s, **BASE_CFG))